# 4-2 条形图与直方图

条形图和直方图都使用矩形展示数据，但它们解决的问题不同。

| 图表 | 横轴含义 | 纵轴含义 | 适用场景 |
|---|---|---|---|
| 条形图 | 离散类别 | 各类别对应的数值，如销量、均值或频数 | 比较不同类别 |
| 直方图 | 连续数值划分出的区间 | 各区间中的样本数，或概率密度 | 观察数值分布 |

> 条形图的高度不一定是频数；它也可以表示金额、数量等其他统计值。

> 约定：使用 `np` 作为 NumPy 的别名，使用 `plt` 作为 `matplotlib.pyplot` 的别名。
> 下方准备单元格会选择可用的中文字体；可以从项目根目录或本章目录运行。
> `fonts` 文件夹是可选的，没有该文件夹时会查找系统字体。
> 如果没有可用的中文字体，会显示提示并使用默认字体继续绘图，中文可能显示为方框。
> 字体设置只作用于当前 Python 进程，详细原理见第 4-6 节。

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

# fonts 文件夹是可选的；没有字体文件时，使用系统已安装的字体。
project_dir = Path('.')
if Path.cwd().name == '第4章-Matplotlib':
    project_dir = Path('..')
font_files = [
    project_dir / 'fonts' / 'NotoSansCJKsc-Regular.otf',
    project_dir / 'fonts' / 'NotoSansSC-Regular.ttf',
]
font_path = next((path for path in font_files if path.is_file()), None)

if font_path is None:
    font_candidates = [
        'Microsoft YaHei', 'SimHei', 'Noto Sans CJK SC', 'Noto Sans SC',
        'Source Han Sans SC', 'PingFang SC', 'Heiti SC', 'Songti SC',
        'Arial Unicode MS',
    ]
    available_fonts = {entry.name for entry in font_manager.fontManager.ttflist}
    font_name = next((name for name in font_candidates if name in available_fonts), None)
    if font_name is None:
        font_name = 'DejaVu Sans'
        print('未找到中文字体，暂用 DejaVu Sans，图中的中文可能显示为方框。')
        print('可安装 Noto Sans CJK SC，或将中文字体文件放入项目的 fonts 文件夹后重新运行。')
    font_path = Path(font_manager.findfont(
        font_manager.FontProperties(family=font_name), fallback_to_default=False
    ))
else:
    font_name = font_manager.FontProperties(fname=str(font_path)).get_name()

font_manager.fontManager.addfont(str(font_path))
plt.rcParams['font.family'] = [font_name]
plt.rcParams['axes.unicode_minus'] = False

import pandas as pd

## 1. 使用 `plt.bar()` 绘制条形图

基本格式为 `plt.bar(类别, 数值)`。下面比较四个类别的数量。

In [ ]:
bar_x = ['a', 'b', 'c', 'd']
bar_y = [20, 10, 30, 25]

plt.figure(figsize=(6, 4))
plt.bar(bar_x, bar_y, color='red')
plt.xlabel('类别')
plt.ylabel('数量')
plt.title('各类别数量对比')
plt.tight_layout()
plt.show()

类别 `c` 的数量最高。横轴上的类别之间不表示连续数值区间，类别顺序可按分析需要调整。

## 2. 使用 `plt.hist()` 绘制直方图

先生成 500 个服从标准正态分布的随机数，使用固定种子保证结果可以复现。
`plt.hist()` 会自动把样本分到不同区间中，并统计每个区间的样本数。

In [ ]:
rng = np.random.default_rng(42)
s = pd.Series(rng.standard_normal(500), name='样本')
s.head()

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(s)
plt.xlabel('样本值')
plt.ylabel('频数')
plt.title('500 个随机样本的分布')
plt.tight_layout()
plt.show()

## 3. 设置直方图的参数

| 参数 | 作用 | 示例 |
|---|---|---|
| `bins` | 分箱数量，也可以传入区间边界 | `5` |
| `rwidth` | 矩形宽度占分箱宽度的比例 | `0.8` |
| `color` | 矩形填充颜色 | `'black'` |
| `edgecolor` | 矩形边框颜色 | `'white'` |

### 3.1 使用 `rwidth` 调整相对宽度

`rwidth=0.8` 表示矩形只占分箱宽度的 80%，会留出可见间隔。
它改变的是外观，不会改变分箱边界或频数。

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(s, rwidth=0.8)
plt.xlabel('样本值')
plt.ylabel('频数')
plt.title('调整直方图的矩形宽度')
plt.tight_layout()
plt.show()

### 3.2 使用 `bins` 设置分箱数

将同一批数据划分成 5 个区间，观察分布会更粗略；分箱过多则可能突出随机波动。

In [ ]:
plt.figure(figsize=(6, 4))
counts, edges, bars = plt.hist(s, bins=5, rwidth=0.8, color='blue')
plt.xlabel('样本值所在区间')
plt.ylabel('频数')
plt.title('将样本划分为 5 个区间')
plt.tight_layout()
plt.show()

`plt.hist()` 返回频数、分箱边界和图形对象。5 个区间对应 6 个边界，所有频数相加为样本总数。

In [ ]:
print('各区间频数：', counts)
print('分箱边界：', edges.round(2))
print('频数合计：', int(counts.sum()))

## 4. 补充：频数与概率密度

默认纵轴是频数。设置 `density=True` 后，纵轴表示概率密度，各分箱的“密度 × 分箱宽度”之和为 `1`。
这不等于各矩形高度之和为 `1`。`rwidth` 只改变显示宽度，留出间隔后可见矩形的面积之和会缩小，但密度计算不变。概率密度使用 `density=True` 设置。

In [ ]:
plt.figure(figsize=(6, 4))
plt.hist(s, bins=15, density=True, color='steelblue', edgecolor='white')
plt.xlabel('样本值')
plt.ylabel('概率密度')
plt.title('样本的概率密度直方图')
plt.tight_layout()
plt.show()

## 5. 要点总结

- 条形图比较不同类别，直方图展示数值样本的分布。
- `plt.bar()` 接收类别与对应数值，`plt.hist()` 接收一批原始样本。
- `bins` 控制分箱，`rwidth` 只控制矩形相对宽度。
- 直方图默认统计频数；`density=True` 将纵轴改为概率密度。
- 选择分箱数时，应兼顾整体形状与细节，避免仅凭一张直方图下结论。